In [ ]:
from pathlib import Path
from datetime import date
import json

from shapely.geometry import box, mapping


In [ ]:
import sys
import os

# Get the path of the project root directory 
# Assumes you are running jupyter lab from inside the 'earth-intelligence-platform' directory
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to the system path so Python can find 'eintelligence'
if project_root not in sys.path:
    sys.path.append(project_root)
    
print(f"Project root added to path: {project_root}")

In [ ]:
from eintelligence.data_prep.aoi import square_aoi
from orchestrator.workflow_manager import (
    DeforestationWorkflow, FloodWorkflow, TilingConfig, TrainingConfig
)

# central california
# aoi = square_aoi(36.3, -120.7)
# print(aoi["geometry"]["coordinates"][0][:2])  # check polygon corners

# munich, germany
aoi = square_aoi(48.1351, 11.5820)
print(aoi["geometry"]["coordinates"][0][:2])  # check polygon corners

In [ ]:
# aoi_geom = mapping(box(-120.7, 36.3, -120.3, 36.7))  # (minx, miny, maxx, maxy)
# aoi_geojson = {"type": "Feature", "geometry": aoi_geom, "properties": {}}


In [ ]:
tiling_cfg = TilingConfig(bands_s2=("B02","B03","B04","B08"), tile_size=512, stride=256, max_cloud=20)
train_cfg  = TrainingConfig(batch_size=8, num_epochs=10, amp=True)

In [ ]:
defor = DeforestationWorkflow(project_root, tiling_cfg, train_cfg, skip_to_pairing=False)

pairs_manifest = defor.build_data(
    aoi_geojson=aoi,
    start="2023-06-01",
    end="2023-08-01",
    region_name="munich"
)

In [ ]:
ckpt_path = Path(project_root) / "models" / "deforestation_resnet18.pt"
out_dir = Path(project_root) / "data" / "munich" / "pred_deforestation"

defor.run_deforestation_workflow(pairs_manifest, ckpt_path, out_dir, retrain=False)

